# V8 · Stage 1.1 — Grouped-split audit

**Overall verdict**: **FAIL — BLOCKS ALL CORPUS-BASED PERFORMANCE CLAIMS AND REQUIRES DATASET REBUILD**

**Qualifier**: Does not demonstrate direct leakage from the three external real cells into training. External-cell metrics require rerunning after clean model selection.

**Roadmap reference**: `paper/research_roadmap.md` · Stage 1 · Task 1.1
**Strategy plan**: `pybamm_research_strengthening_plan.md`
**Follow-on notebooks (added on this verdict)**:
`01_1b_rebuild_grouped_dataset.ipynb` → `01_1c_retrain_clean_split.ipynb` → `01_1d_compare_leaked_vs_clean.ipynb`

## Acceptance criterion
Zero cross-split leakage on `sim_id`; anchor-grouped v6/v7 splits report identical
anchor→split map.

## Recommended status statement (embed verbatim in the V8 abstract and any release note)
> The initial v7 corpus used row-level random splitting. A grouped-split audit found that 483 of 486 simulated trajectories crossed train, validation and test partitions, and 76.1% of base context windows had SoH-offset augmentation copies distributed across multiple splits. Additional temporal overlap showed that validation contexts could contain cycles previously observed as training targets. Consequently, corpus validation loss, checkpoint selection and synthetic test metrics are considered invalid. The external real-cell evaluations were not directly included in the corpus, but they will be repeated using a model retrained under trajectory-grouped splitting.


## 1. Setup

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJ = Path("/home/hj/Desktop/PINNs")
sys.path.insert(0, str(PROJ / "Voltaris" / "Data_Exploration"))
from phase3_v7_train import _stratified_split, _compute_normalisation, SEED as SPLIT_SEED

DATASET = PROJ / "configs" / "phase3_corpus" / "_v7_dataset.parquet"
OUT_JSON = PROJ / "outputs" / "results" / "grouped_split_audit_raw.json"
OUT_JSON.parent.mkdir(parents=True, exist_ok=True)

print(f"split seed = {SPLIT_SEED}")
print(f"dataset    = {DATASET}")
print(f"output     = {OUT_JSON}")


split seed = 456
dataset    = /home/hj/Desktop/PINNs/configs/phase3_corpus/_v7_dataset.parquet
output     = /home/hj/Desktop/PINNs/outputs/results/grouped_split_audit_raw.json


## 2. Method

Deterministic pandas set-operations on the 14,096-row v7 dataset. For each
dimension we group rows and count how many groups appear in more than one of
`{train, val, test}`.

The splitter is `_stratified_split(df, seed=456)` — the exact function used by
`phase3_v7_train.train_v7`, so the audit reproduces production behaviour.

A leak-free split under proper grouping would report `n_leaked_groups == 0` on
every dimension EXCEPT `anchor_id`, which is deliberately stratified.


In [2]:
# Load dataset + apply canonical splitter
df = pd.read_parquet(DATASET)
tr, va, te = _stratified_split(df, seed=SPLIT_SEED)
sp = pd.Series(index=df.index, dtype="object")
sp.loc[tr] = "train"; sp.loc[va] = "val"; sp.loc[te] = "test"
df["split"] = sp.values

print(f"loaded {len(df):,} rows")
print(df["split"].value_counts())


loaded 14,096 rows
split
train    9868
val      2114
test     2114
Name: count, dtype: int64


In [3]:
# Per-dimension leakage counts
def leak_dim(cols, name):
    per = df.groupby(cols)["split"].agg(lambda s: tuple(sorted(set(s))))
    leaked_keys = per[per.apply(len) > 1]
    total = len(per)
    n_leaked = len(leaked_keys)
    if isinstance(cols, str):
        mask = df[cols].isin(leaked_keys.index)
    else:
        idx = pd.MultiIndex.from_frame(df[list(cols)])
        mask = idx.isin(leaked_keys.index)
    return {
        "dimension": name,
        "n_groups": int(total),
        "n_leaked_groups": int(n_leaked),
        "pct_leaked": round(100 * n_leaked / max(1, total), 3),
        "n_rows_in_leaked_groups": int(mask.sum()),
    }

r_anchor = leak_dim("anchor_id", "anchor_id")
r_sim = leak_dim(["anchor_id", "sample_id"], "sim_id (anchor_id + sample_id)")
r_ctx = leak_dim(["anchor_id", "sample_id", "context_start"],
                 "base_window (anchor + sample + context_start)")

per_dim = [r_anchor, r_sim, r_ctx]
pd.DataFrame(per_dim)


,dimension,n_groups,n_leaked_groups,pct_leaked,n_rows_in_leaked_groups
0,anchor_id,7,7,100.000,14096
1,sim_id (anchor_id + sample_id),486,483,99.383,14080
2,base_window (anchor + sample + context_start),3524,2682,76.107,10728


In [4]:
# Temporal overlap — ordered (i, j) pairs within the same sim, different splits
target_ranges = df.apply(
    lambda r: (int(r["context_start"]), int(r["context_start"]) + int(r["K"]),
               int(r["context_start"]) + int(r["K"]),
               int(r["context_start"]) + int(r["K"]) + int(r["forecast_len"])),
    axis=1
)
df["ctx_lo"] = [t[0] for t in target_ranges]
df["ctx_hi"] = [t[1] for t in target_ranges]
df["tgt_lo"] = [t[2] for t in target_ranges]
df["tgt_hi"] = [t[3] for t in target_ranges]

temporal_leaks = 0
temporal_examples = []
for (aid, sid), grp in df.groupby(["anchor_id", "sample_id"]):
    grp = grp.sort_values("context_start").reset_index(drop=True)
    if grp["split"].nunique() < 2:
        continue
    for i in range(len(grp)):
        for j in range(len(grp)):
            if i == j: continue
            if grp.loc[i, "split"] == grp.loc[j, "split"]: continue
            if grp.loc[i, "tgt_lo"] < grp.loc[j, "ctx_hi"] and grp.loc[j, "ctx_lo"] < grp.loc[i, "tgt_hi"]:
                temporal_leaks += 1
                if len(temporal_examples) < 3:
                    temporal_examples.append({
                        "sim": [str(aid), str(sid)],
                        "row_i_split": grp.loc[i, "split"],
                        "row_i_target_cycles": [int(grp.loc[i, "tgt_lo"]), int(grp.loc[i, "tgt_hi"])],
                        "row_j_split": grp.loc[j, "split"],
                        "row_j_context_cycles": [int(grp.loc[j, "ctx_lo"]), int(grp.loc[j, "ctx_hi"])],
                    })
print(f"temporal leaked ordered pairs: {temporal_leaks:,}")
print("first 3 examples:")
for ex in temporal_examples:
    print(" ", ex)


temporal leaked ordered pairs: 29,187
first 3 examples:
  {'sim': ['CALB_0003', 'CALB_0003_s0000'], 'row_i_split': 'train', 'row_i_target_cycles': [50, 450], 'row_j_split': 'val', 'row_j_context_cycles': [100, 150]}
  {'sim': ['CALB_0003', 'CALB_0003_s0000'], 'row_i_split': 'train', 'row_i_target_cycles': [50, 450], 'row_j_split': 'test', 'row_j_context_cycles': [100, 150]}
  {'sim': ['CALB_0003', 'CALB_0003_s0000'], 'row_i_split': 'train', 'row_i_target_cycles': [50, 450], 'row_j_split': 'test', 'row_j_context_cycles': [300, 350]}


## 3. Normalisation-fit leakage audit

**Verdict**: **PASS — no normalisation-fit leakage detected.**

Feature-normalisation statistics were computed exclusively from the training
partition. Train-only and full-corpus means and standard deviations differed by
less than $10^{-4}$ for all non-constant health features, confirming that
preprocessing-statistic leakage is absent and unlikely to materially affect
results. Nevertheless, trajectory-level, augmentation-group, and temporal
target–context leakage remain and invalidate the independence of the corpus
validation and test partitions.

| Statistic | Where computed | Leakage risk |
|---|---|---|
| `x_health` z-score (T, c_rate, DCIR) | `_compute_normalisation(df.iloc[train_idx])` | none — fit on training rows only |
| `theta_norm` z-score | Pre-standardised in extractor using the anchor's OWN DE-fit base value + phase3_sweep σ; NOT fit on the row dataset | none — anchor-parametric, not dataset-dependent |
| Corpus SoH normalisation | Each simulated trajectory divides by its OWN cycle-1 capacity | none — per-sim, not dataset-fit |
| SoH-offset augmentation ranges | Hardcoded `[0, -0.1, -0.2, -0.3]` | none — no fit |

### Temperature feature note
Temperature was constant at 25 °C throughout the corpus; its standard deviation
was replaced with 1 during normalisation (guard in `_compute_normalisation`),
making the normalised temperature feature identically zero. That feature
currently carries no predictive information; it can remain for future
compatibility but should not be described as an active operating-condition input
in the current dataset.


In [5]:
# Reproduce this check with the actual normalisation function
norm_train_only = _compute_normalisation(df.iloc[tr])
norm_full = _compute_normalisation(df)   # what would happen if fit on full df
xh_mean_train = norm_train_only["xh_mean"].numpy()
xh_mean_full  = norm_full["xh_mean"].numpy()
xh_std_train  = norm_train_only["xh_std"].numpy()
xh_std_full   = norm_full["xh_std"].numpy()

print("x_health mean (train-only vs full-dataset):")
print(f"  train: {xh_mean_train}")
print(f"  full : {xh_mean_full}")
print(f"  |Δ|  : {np.abs(xh_mean_train - xh_mean_full)}")

print("x_health std (train-only vs full-dataset):")
print(f"  train: {xh_std_train}")
print(f"  full : {xh_std_full}")
print(f"  |Δ|  : {np.abs(xh_std_train - xh_std_full)}")
print()
print("Confirmed: v7 code fits stats on training rows only "
      "(no normalisation-fit leakage).")


x_health mean (train-only vs full-dataset):
  train: [25.         0.3961289  1.0397259]
  full : [25.          0.39614075  1.0396531 ]
  |Δ|  : [0.0000000e+00 1.1861324e-05 7.2836876e-05]
x_health std (train-only vs full-dataset):
  train: [1.        0.123199  0.1644001]
  full : [1.         0.12320284 0.16440965]
  |Δ|  : [0.000000e+00 3.837049e-06 9.551644e-06]

Confirmed: v7 code fits stats on training rows only (no normalisation-fit leakage).


## 4. Results — audit summary

| Audit item | Verdict |
|---|---|
| Scaler fitted using training rows only | **PASS** |
| Difference between train-only and full statistics | **Negligible** |
| `sim_id` isolated across splits | **FAIL** |
| Augmentation copies isolated across splits | **FAIL** |
| Temporal target–context independence | **FAIL** |
| Corpus validation/test metrics trustworthy | **NO** |
| Anchor stratification (7/7 in every split) | **PASS WITH LIMITATIONS** — valid only for within-anchor interpolation |

### Per-dimension detail

| # | Dimension | Metric | Verdict | Reason |
|---|---|---|---|---|
| 1 | anchor_id | 7 / 7 anchors (100%) span all splits. | **PASS_WITH_LIMITATIONS** | Intentional within-anchor stratification. Acceptable ONLY for within-anchor interpolation; invalid for anchor-generalisation or unseen-anchor claims. |
| 2 | sim_id (anchor_id + sample_id) | 483 / 486 sims (99.4%) span multiple splits; 14,080 of 14,096 rows in leaked groups. | **FAIL** | Fatal trajectory-level leakage. Val loss, corpus test loss, checkpoint selection, early-stopping epoch and hyperparameter decisions based on val performance are NOT trustworthy. |
| 3 | base_window (anchor + sample + context_start) | 2,682 / 3,524 base windows (76.1%) have SoH-offset copies split across train/val/test. | **FAIL** | The 4 SoH-offset variants of a single base window are near-duplicates; they must live in the same split. |
| 4 | SoH-offset augmentation copies | Same 76.1% base windows have their 4 aug copies scattered — the aug-grouping ML rule is violated. | **FAIL** | Aug-copies with different offsets landing in different splits let val/test rows become trivial +offset transforms of training rows. |
| 5 | temporal target-context overlap | 29,187 ordered (i, j) pairs where row_i's target cycles (train) overlap row_j's context cycles (val/test) within the SAME simulation. | **FAIL** | Even base-window grouping alone would be insufficient. Split must occur at the COMPLETE trajectory level, not the window level. |

Each `FAIL` verdict has been adversarially challenged — best-effort defence
attempted; defence failed. The temporal-overlap dimension confirms that
window-level grouping alone would still leak; the split must occur at the
**complete trajectory level**.


In [6]:
# Persist raw metrics for downstream consumers
verdicts_json = [
    {"dimension": v[0], "metric": v[1], "verdict": v[2], "reason": v[3]}
    for v in [
        ("anchor_id", "7 / 7 anchors (100%) span all splits.",
         "PASS_WITH_LIMITATIONS",
         "Intentional within-anchor stratification. Acceptable ONLY for within-anchor interpolation."),
        ("sim_id (anchor_id + sample_id)", "483 / 486 sims (99.4%) span multiple splits.",
         "FAIL", "Fatal trajectory-level leakage."),
        ("base_window (anchor + sample + context_start)",
         "2,682 / 3,524 base windows (76.1%) crossed splits.",
         "FAIL", "SoH-offset copies must live in the same split."),
        ("SoH-offset augmentation copies",
         "Same 76.1% base windows split their aug copies.",
         "FAIL", "Aug-copies are near-duplicates; must be grouped."),
        ("temporal target-context overlap",
         f"{temporal_leaks:,} ordered (i,j) pairs",
         "FAIL", "Split must occur at complete trajectory level."),
    ]
]

out = {
    "split_seed": SPLIT_SEED,
    "n_rows": len(df),
    "split_counts": df["split"].value_counts().to_dict(),
    "dimensions": per_dim + [{
        "dimension": "temporal target-context overlap between splits",
        "n_leaked_pairs": int(temporal_leaks),
        "example_leaked_pairs": temporal_examples,
    }],
    "verdicts": verdicts_json,
    "overall_verdict": "FAIL — BLOCKS ALL CORPUS-BASED PERFORMANCE CLAIMS AND REQUIRES DATASET REBUILD",
    "qualifier": "Does not demonstrate direct leakage from the three external real cells into training. External-cell metrics require rerunning after clean model selection.",
}
OUT_JSON.write_text(json.dumps(out, indent=2, default=str))
print(f"wrote {OUT_JSON}")


wrote /home/hj/Desktop/PINNs/outputs/results/grouped_split_audit_raw.json


## 5. Downstream consequences

Because the v7 dataset splits leak at the trajectory level:

### NOT trustworthy
- v7.1 val loss (used for early-stopping)
- v7.1 corpus test loss
- v7.1 checkpoint selection
- v7.1 early-stopping epoch
- v7.1 hyperparameter decisions based on val performance

### NOT directly contaminated, but MUST be rerun with a clean checkpoint
- **The 0.96 / 0.79 / 0.47 pp held-out RMSE** on CALB_0029, EVE_0003, REPT_0028.
- These external cells were never in the corpus splits, so they are not
  contaminated by direct sample leakage.
- BUT the reported model was selected using a leaked synthetic validation
  split. Their errors should be re-evaluated after retraining under a
  trajectory-grouped split.

### Blocked V8 notebooks (must wait for clean dataset + clean checkpoint)
- `01_2_baselines_linear_exp` — needs a clean split for a fair baseline comparison
- `01_3_no_theta_ablation` — no-θ vs v7.1 comparison meaningless while both use leaked splits
- `01_5_multi_seed_variance` — same reason
- `02_1_leave_one_anchor_out`, `02_2_leave_one_supplier_out`, `02_4_context_length_study`
  — these are the correct remedies but require the clean splitter first

### Newly required notebooks (added on this verdict)
1. **`01_1b_rebuild_grouped_dataset.ipynb`** — build a trajectory-grouped
   splitter. Split at complete `sim_id` level FIRST, then generate windows
   inside each assigned split, then apply SoH-offset augmentation inside the
   same split. Hard assertions on:
   ```python
   assert df.groupby(["anchor_id", "sample_id"])["split"].nunique().max() == 1
   assert df.groupby(["anchor_id", "sample_id", "context_start"])["split"].nunique().max() == 1
   ```
   plus a temporal-overlap re-audit that must return 0.

2. **`01_1c_retrain_clean_split.ipynb`** — retrain from scratch on the clean
   dataset. Do NOT continue training the v7.1 checkpoint. Refit
   normalisation stats on training rows only. Report new val/test RMSE.

3. **`01_1d_compare_leaked_vs_clean.ipynb`** — side-by-side comparison of the
   frozen `v7_leaked_split` checkpoint vs the new clean checkpoint, on:
   - corpus val/test RMSE (expected to worsen after the fix)
   - held-out real-cell RMSE (expected to be re-evaluated)
   - training-curve shape
   - epoch of best val
   - no-θ ablation (rerun on both models)

### Freeze the current model
Do NOT overwrite `outputs/models/pinn_phase3_v7_1_operator.pt`. Keep the
weights, config, split assignments, metrics and plots for reproducible
comparison. Add a `v7_leaked_split` alias if useful.

### Expected outcome after fixing the split
- Val loss will increase
- Larger epoch-to-epoch variance
- Earlier or later optimal stopping
- Wider train-val gap
- Real-cell RMSE may worsen
- No-θ ablation results become more informative

**This is not a failure. It is the first credible estimate of model performance.**


## 6. Verdict marker

- [ ] **PASS** — acceptance criterion fully met.
- [ ] **PASS WITH LIMITATIONS** — criterion met with a named caveat.
- [x] **FAIL — BLOCKS ALL CORPUS-BASED PERFORMANCE CLAIMS AND REQUIRES DATASET REBUILD**

**Qualifier**: Does not demonstrate direct leakage from the three external real cells into training. External-cell metrics require rerunning after clean model selection.

**Result artifact(s) written**: `outputs/results/grouped_split_audit_raw.json`

**Notes**: Five leakage dimensions audited; four FAIL (sim_id 99.4%, base_window
76.1%, SoH-aug 76.1%, temporal 29,187 pairs), one PASS_WITH_LIMITATIONS
(anchor_id — intentional stratification, valid only for within-anchor
interpolation). Fix path is a `GroupKFold` split at complete `sim_id` level
BEFORE window generation, plus base-window grouping for augmentation copies.
See section 5 for the three new blocking notebooks (`01_1b`, `01_1c`, `01_1d`).
